# سبق 18 (تسلسل): رسیدیں جو ثابت کرتی ہیں کہ ایک *انسانی* نے کارروائی کی اجازت دی

یہ سبق اس بات کا ثبوت دیتا ہے کہ **ایجنٹ** نے کیا کیا اور **گیٹ** نے کیا فیصلہ کیا۔ یہ نوٹ بک وہ گمشدہ نصف اضافہ کرتی ہے: ثبوت کہ ایک **نامزد انسان** نے **عین** کارروائی کی منظوری دی — ایک الگ، انسانی دستخط جو مکمل معیاری کارروائی پر ہوتا ہے، آف لائن تصدیق شدہ۔

یہاں دونوں آرٹیفیکٹس سبق کی رسیدوں کے **اسی لفافہ شکل** استعمال کرتے ہیں: ایک سیدھے سادے پے لوڈ کے ساتھ ایک `type` فیلڈ، جس پر Ed25519 کے ذریعہ معیاری پے لوڈ بائٹس کے SHA-256 پر دستخط ہوتے ہیں، اور ایک منظم `signature` آبجیکٹ منسلک ہوتا ہے (جو دستخط شدہ بائٹس سے خارج کیا گیا ہے)۔ منظوری کی رسید ایک نیا `type` (`human.approval.v1`) ہے جو کارروائی کی قسم کے ساتھ آتا ہے، لہٰذا ایک `verify_chain` دونوں آرٹیفیکٹ اقسام کو کور کرتا ہے اسی کوڈ راستے سے جو آپ نے مرکزی نوٹ بک میں بنایا تھا۔ یہ وہی شکل ہے جو سبق انٹرنیٹ ڈرافٹ (draft-farley-acta-signed-receipts) میں کو-دستخط کے راستے کی بھی ہے۔

مرکزی نوٹ بک کے ڈیمو تصدیق کنندہ پر ایک جان بوجھ کر اپ گریڈ: یہاں تصدیق کنندہ `signature.key_id` کو رسید کے اندر موجود پبلک کی پر بھروسہ کرنے کے بجائے **پِن کی گئی کلید کی رجسٹری** سے حل کرتا ہے۔ یہ وہ پروڈکشن رویہ ہے جس کی سبق کی اپنی چیک لسٹ تجویز کرتی ہے ("تصدیق کی پبلک کی شائع کریں")، اور یہی وہ چیز ہے جو جعلسازی کو ایک انکار بناتی ہے نہ کہ اپنا کی لانے کی گریز۔

یہ نوٹ بک جو اصول سکھاتی ہے: **ایک دستخط شدہ منظوری خود میں اختیار نہیں ہے۔** اختیار صرف اسی وقت موجود ہوتا ہے جب منظوری کی رسید اور کارروائی کی رسید دونوں عملدرآمد کے وقت همان معیاری کارروائی سے بندھے ہوں، ایک موجودہ پالیسی ورژن، کلید، اور میعاد ختم ہونے کے تحت، اور ایک ایسی منظوری جو پہلے سے استعمال نہ ہوئی ہو۔ ہر ناکامی ایک **مخصوص وجہ** سے انکار کرتی ہے، تاکہ آپ بیان کر سکیں کہ *اختیار متروک ہو گیا* ہے یا *عملدرآمد شدہ کارروائی تبدیل ہو گئی* ہے۔


In [1]:
# These are already the Lesson 18 dependencies — no new packages.
# %pip install pynacl jcs
import base64, copy, hashlib
from jcs import canonicalize                      # RFC 8785 canonical JSON
from nacl.signing import SigningKey, VerifyKey
# CryptoError is the common base of BadSignatureError AND the ValueError pynacl
# raises for a wrong-length signature — catch the base so verification fails
# closed on ANY bad signature, not just the forged-but-correct-length one.
from nacl.exceptions import CryptoError

# Same helpers as the main notebook.
def b64url_nopad(data: bytes) -> str:
    return base64.urlsafe_b64encode(data).decode("ascii").rstrip("=")

def b64url_decode(s: str) -> bytes:
    return base64.urlsafe_b64decode(s + "=" * ((4 - len(s) % 4) % 4))

def sha256_canonical(obj) -> str:
    """SHA-256 of an object's JCS-canonical JSON form (same helper as the lesson)."""
    return f"sha256:{hashlib.sha256(canonicalize(obj)).hexdigest()}"

## درست عمل

منظوری کی اکائی **کینونیکل ایکشن آبجیکٹ** ہے — محض ایک مبہم لیبل جیسے "رقم کی واپسی کی منظوری" نہیں، بلکہ بالکل مخصوص مکمل طور پر متعین عمل۔ پورے آبجیکٹ پر دستخط کرنا (اور اس سے ایک ہضم نکالنا) ہمیں بعد میں یہ ثابت کرنے کی اجازت دیتا ہے کہ انسان نے *یہی* منظور کیا اور کچھ نہیں۔


In [2]:
action = {
    "action_type": "refund.issue",
    "params": {"order_id": "A-1029", "amount_usd": 4200, "to": "acct_88"},
    "policy_id": "refunds-v3",
}
print("action digest:", sha256_canonical(action))

action digest: sha256:fba342ad8447b491a089d7a09d4ac58f1a835c504e58f8d832db04f65bb62a25


## ایک لفافہ، دو اختیارات

ہر رسید سبق کا لفافہ ہے: ایک سیدھی سی معلومات جس میں `type` فیلڈ ہوتی ہے، اس کے علاوہ ایک `signature` آبجیکٹ (`alg`, `sig`, `key_id`) ہوتا ہے جو دستخط کیے گئے بائٹس کا حصہ **نہیں** ہوتا۔ `verify_envelope` دونوں قسم کی رسیدوں کے لیے مشترکہ ساختی اور دستخطی چیک ہے؛ جس **پین کردہ کلید رجسٹری** کی بنیاد پر یہ `signature.key_id` کو حل کرتا ہے وہ اختیارات کو الگ رکھتی ہے:

- **منظوری رسید** (`human.approval.v1`) — نامزد منظور کنندہ، مکمل معیاری عمل **اور اس کا خلاصہ**، `policy_version`, اجرا + اختتامی ٹائم اسٹیمپس۔ ایک بار استعمال کرنا چین سطح پر ٹریک کیا جاتا ہے۔
- **عمل کی رسید** (`agent.action.v1`) — ایجنٹ کی شناخت، `run_id`, وہی معیاری عمل **خلاصہ**، عمل کا نتیجہ + ٹائم اسٹیمپ، اور `parent_approval_ref`: منظوری کا `receipt_hash`, وہی کنونشن جو سبق کی چین میں `previous_receipt_hash` کے لیے ہے۔

مشترکہ `action_digest` فیلڈ وہ جوڑ ہے جس پر بندھن منحصر ہے۔ `key_id` دستخط کے آبجیکٹ میں صرف ایک تلاش کی نشانی کے طور پر موجود ہے: اسے کسی مختلف پین کلید کی طرف موڑنے سے دستخطی چیک ناکام ہو جاتا ہے، تو یہ کچھ نہیں دیتا۔


In [3]:
# ---- pinned key registries: SEPARATE authorities, one envelope shape ----------
# Published out of band (the lesson checklist's JWK-Set pattern); the verifier
# NEVER trusts a key carried inside a receipt.
approver_sk = SigningKey.generate()
agent_sk    = SigningKey.generate()
APPROVER_KEYS = {"approver-key-1": b64url_nopad(bytes(approver_sk.verify_key))}
AGENT_KEYS    = {"agent-key-1":    b64url_nopad(bytes(agent_sk.verify_key))}

# The policy the approval is granted under. If this moves after approval, the
# approval is STALE even though its signature still verifies.
CURRENT_POLICY = {"policy_version": "refunds-v3"}

def sign_receipt(payload: dict, sk: SigningKey, key_id: str) -> dict:
    """Same signing pipeline as the lesson: Ed25519 over the SHA-256 of the
    canonical payload; the signature object is NOT part of the signed bytes."""
    message_hash = hashlib.sha256(canonicalize(payload)).digest()
    return {
        **payload,
        "signature": {"alg": "EdDSA", "sig": b64url_nopad(sk.sign(message_hash).signature), "key_id": key_id},
    }

def verify_envelope(receipt, expected_type: str, trusted_keys: dict):
    """The SHARED verifier contract for any receipt kind; the caller picks which
    pinned registry (authority) resolves key_id. Fails closed on ANY
    attacker-shaped input: malformed is a refusal, never a crash."""
    if not isinstance(receipt, dict) or not isinstance(receipt.get("signature"), dict):
        return (False, "receipt malformed (not an object with a signature object)")
    sig_obj = receipt["signature"]
    if sig_obj.get("alg") != "EdDSA":
        return (False, "unsupported signature alg")
    if receipt.get("type") != expected_type:
        return (False, f"wrong receipt type (expected {expected_type})")
    # Key freshness is part of authority: a key_id rotated out of the pinned
    # registry confers nothing, even with a valid signature.
    pub = trusted_keys.get(sig_obj.get("key_id"))
    if pub is None:
        return (False, f"stale authority: key_id {sig_obj.get('key_id')!r} is not in the pinned registry (unknown or rotated out)")
    # Reconstruct the signed bytes exactly as the lesson does: everything except
    # the signature object, canonicalized, hashed.
    payload = {k: v for k, v in receipt.items() if k != "signature"}
    try:
        message_hash = hashlib.sha256(canonicalize(payload)).digest()
        VerifyKey(b64url_decode(pub)).verify(message_hash, b64url_decode(sig_obj.get("sig") or ""))
    except (CryptoError, TypeError, ValueError, base64.binascii.Error):
        return (False, "signature invalid (forged, tampered, or malformed)")
    return (True, "envelope ok")

def human_approval(action, approver_id, approved_at, sk=approver_sk,
                   key_id="approver-key-1", policy_version=None, expires_at=None):
    # deepcopy: the receipt must be an immutable record of what was approved —
    # a live reference would let a later mutation of `action` silently change the
    # signed payload. Digest the SNAPSHOT so the two can never diverge.
    approved_action = copy.deepcopy(action)
    payload = {
        "type": "human.approval.v1",
        "approver_id": approver_id,
        "action": approved_action,                       # the FULL canonical action
        "action_digest": sha256_canonical(approved_action),  # the join field
        "policy_version": policy_version or CURRENT_POLICY["policy_version"],
        "approved_at": approved_at,                      # ISO-8601 Zulu, like the lesson
        "expires_at": expires_at or approved_at[:11] + "23:59:59Z",
    }
    return sign_receipt(payload, sk, key_id)

In [4]:
approval = human_approval(action, "alice@ops (WebAuthn)", "2026-07-08T15:04:05Z",
                          expires_at="2026-07-08T15:19:05Z")
print(verify_envelope(approval, "human.approval.v1", APPROVER_KEYS))
print("binds digest:", approval["action_digest"][:23], "…  under", approval["policy_version"])

(True, 'envelope ok')
binds digest: sha256:fba342ad8447b491 …  under refunds-v3


## `verify_chain`: وہ جگہ جہاں بندھن درحقیقت فیصلہ کیا جاتا ہے  

`verify_chain` دو دستخطی جانچوں کا آسان لفافہ **نہیں** ہے۔ یہ ایک ہی جگہ ہے جہاں مشترکہ معیاری `action_digest`، منظوری کی پالیسی/کلید/اختتامی تاریخ کی **تازگی**، اور منظوری کی **ایک بار استعمال ہونے** کی جانچ ایک ساتھ کی جاتی ہے، اس عمل کے خلاف جو *ابھی ابھی* انجام دیا جا رہا ہے۔  

ہر ناکامی ایک **مختلف وجہ** کے ساتھ منع کرتی ہے، تاکہ منع کرنے والا پڑھنے والا یہ سمجھ سکے کہ اختیار غیر موثر ہو گیا (پالیسی تبدیل ہو گئی، کلید گھمایا گیا، منظوری کی میعاد ختم ہو گئی، منظوری استعمال ہو گئی) یا انجام دیے جانے والے عمل نے ابھی بھی درست منظوری کے نیچے تبدیلی کی ہے (ڈائجسٹ کی تبدیلی)۔  


In [5]:
def receipt_hash(receipt: dict) -> str:
    """Content-derived id of a COMPLETE receipt (including its signature) —
    the same convention as previous_receipt_hash in the lesson's chain."""
    return sha256_canonical(receipt)

def agent_receipt(action, approval, executed_at, sk=agent_sk, key_id="agent-key-1"):
    executed_action = copy.deepcopy(action)    # snapshot, same reason as the approval
    payload = {
        "type": "agent.action.v1",
        "agent_id": "agent:refunds-bot",
        "run_id": "run-0001",
        "action": executed_action,
        "action_digest": sha256_canonical(executed_action),  # same join field
        "parent_approval_ref": receipt_hash(approval),
        "outcome": "performed",
        "executed_at": executed_at,
    }
    return sign_receipt(payload, sk, key_id)

_consumed = set()

def verify_chain(action_being_executed, approval, agent_rcpt, now: str):
    """One code path covers both receipt kinds (same envelope), then checks the
    things that only make sense TOGETHER: shared digest, freshness, consumption.
    `now` is an ISO-8601 Zulu timestamp; Zulu strings compare correctly as strings."""
    # 1. Shared envelope contract, separate authorities.
    ok, why = verify_envelope(approval, "human.approval.v1", APPROVER_KEYS)
    if not ok: return (False, f"approval: {why}")
    ok, why = verify_envelope(agent_rcpt, "agent.action.v1", AGENT_KEYS)
    if not ok: return (False, f"agent receipt: {why}")

    # 2. The join: BOTH receipts must bind the digest of the action being executed
    #    right now. A valid approval for a DIFFERENT action is substitution, and it
    #    gets its own reason — this is "the executed action changed".
    executing_digest = sha256_canonical(action_being_executed)
    if approval.get("action_digest") != executing_digest or approval.get("action") != action_being_executed:
        return (False, "digest substitution: the approval binds a different canonical action than the one being executed")
    if agent_rcpt.get("action_digest") != executing_digest or agent_rcpt.get("action") != action_being_executed:
        return (False, "digest substitution: the agent receipt binds a different canonical action than the one being executed")
    if agent_rcpt.get("parent_approval_ref") != receipt_hash(approval):
        return (False, "agent receipt is not bound to this approval")

    # 3. Freshness: a valid signature over stale authority is still a refusal —
    #    each staleness gets its own reason, distinct from substitution above.
    if approval.get("policy_version") != CURRENT_POLICY["policy_version"]:
        return (False, f"stale authority: approved under policy {approval.get('policy_version')!r}, current is {CURRENT_POLICY['policy_version']!r}")
    expires = approval.get("expires_at")
    if not isinstance(expires, str) or not expires or now >= expires:
        return (False, "stale authority: approval expired before execution")

    # 4. One-time consumption: an approval authorizes ONE execution.
    ref = receipt_hash(approval)
    if ref in _consumed:
        return (False, "approval already consumed (replay refused)")
    _consumed.add(ref)
    return (True, f"approved by {approval['approver_id']}, executed by {agent_rcpt['agent_id']}")

def execute(action, approval, agent_rcpt, now):
    ok, why = verify_chain(action, approval, agent_rcpt, now)
    return (ok, "executed" if ok else why)

receipt = agent_receipt(action, approval, "2026-07-08T15:04:06Z")
print(execute(action, approval, receipt, now="2026-07-08T15:04:07Z"))

(True, 'executed')


## وہ کیا چیز بندی پکڑتی ہے

نیچے ہر کیس **مخصوص وجہ** کے ساتھ **بند** ہو جاتا ہے۔ پہلا بلاک کلاسک سیٹ ہے (مداخلت، الجھا ہوا نائب، دوبارہ چلانا، کسی بھی اختیار پر جعلسازی، خراب شدہ ان پٹ)۔ دوسرا بلاک وہ جوڑے ہے جو پراپرٹی کو دعویٰ شدہ کی بجائے حقیقی بناتا ہے:

- **پرانی اتھارٹی** — دستخط ابھی بھی معتبر ہے، لیکن پالیسی کا ورژن تبدیل ہو گیا ہے، منظوری دینے والے کی چابی پنڈ رجسٹری سے ہٹا دی گئی ہے، یا منظوری نفاذ سے پہلے ختم ہو گئی ہے؛
- **ہضم کی جگہ تبدیلی** — ایک درست طریقے سے دستخط شدہ کارروائی کی رسید جس کا `parent_approval_ref` ایک *حقیقی* منظوری کی طرف اشارہ کرتا ہے، لیکن اس منظوری کا کینونیکل کارروائی ہضم اس کارروائی سے میل نہیں کھاتا جو اصل میں نافذ ہو رہی ہے۔


In [6]:
NOW = "2026-07-08T15:05:00Z"

# 1. tamper: change the amount after approval — the executed action changed.
tampered = {**action, "params": {**action["params"], "amount_usd": 9900}}
print("tamper              ->", verify_chain(tampered, approval, agent_receipt(tampered, approval, NOW), NOW))

# 2. confused deputy: valid approval for action A, presented to execute action B.
action_b = {**action, "action_type": "wire.send"}
print("confused-deputy     ->", verify_chain(action_b, approval, agent_receipt(action_b, approval, NOW), NOW))

# 3. replay: the approval was consumed by the successful execution above.
print("replay              ->", execute(action, approval, agent_receipt(action, approval, NOW), NOW))

# 4. forged approval: attacker signs with their own key but claims a pinned key_id.
mallory_sk = SigningKey.generate()
forged = human_approval(action, "mallory", NOW, sk=mallory_sk)
print("forged-approval     ->", verify_chain(action, forged, agent_receipt(action, forged, NOW), NOW))

# A fresh, un-consumed approval so the agent-side cases fail on their OWN check.
fresh = human_approval(action, "alice@ops (WebAuthn)", NOW, expires_at="2026-07-08T15:20:00Z")

# 5. self-minted agent receipt: attacker's own agent key, refused by the pinned registry.
mallory_agent = agent_receipt(action, fresh, NOW, sk=SigningKey.generate())
print("self-minted-agent   ->", verify_chain(action, fresh, mallory_agent, NOW))

# 6. wrong-action agent receipt: real agent key, but the receipt binds a different action.
wrong_action = {**action, "params": {**action["params"], "amount_usd": 9900}}
print("wrong-action-agent  ->", verify_chain(action, fresh, agent_receipt(wrong_action, fresh, NOW), NOW))

# 7. malformed input: structurally broken receipts refuse cleanly, they never crash.
print("malformed-approval  ->", verify_chain(action, {"type": "human.approval.v1"}, agent_receipt(action, fresh, NOW), NOW))
print("malformed-agent     ->", verify_chain(action, fresh, {"nope": "not a receipt"}, NOW))

# 8. wrong-length signature: valid base64, not 64 bytes — refused, not crashed.
badlen = {**fresh, "signature": {**fresh["signature"], "sig": "AAAA"}}
print("wrong-len-sig       ->", verify_chain(action, badlen, agent_receipt(action, fresh, NOW), NOW))

# 9. non-object receipt: a list refuses cleanly instead of raising AttributeError.
print("nonobject-receipt   ->", verify_chain(action, [1, 2], agent_receipt(action, fresh, NOW), NOW))

print()
print("--- the two negative controls that make the property real ---")

# 10. STALE POLICY: signature still valid, but policy moved between approval and
#     execution. Authority is decided at execution time, not signing time.
CURRENT_POLICY["policy_version"] = "refunds-v4"
print("stale-policy        ->", verify_chain(action, fresh, agent_receipt(action, fresh, NOW), NOW))
CURRENT_POLICY["policy_version"] = "refunds-v3"   # restore for the cases below

# 11. STALE KEY: the approver key is rotated out of the pinned registry after
#     signing. The signature bytes still verify against the old key — but the old
#     key no longer confers authority.
rotated_out = APPROVER_KEYS.pop("approver-key-1")
print("stale-key           ->", verify_chain(action, fresh, agent_receipt(action, fresh, NOW), NOW))
APPROVER_KEYS["approver-key-1"] = rotated_out     # restore

# 12. EXPIRED: approval was valid when signed, but execution came too late.
expired = human_approval(action, "alice@ops (WebAuthn)", "2026-07-08T14:00:00Z",
                         expires_at="2026-07-08T14:01:00Z")
print("expired-approval    ->", verify_chain(action, expired, agent_receipt(action, expired, NOW), NOW))

# 13. DIGEST SUBSTITUTION: a validly signed agent receipt whose parent_approval_ref
#     points at a REAL approval — but that approval binds action B, and the agent
#     is executing action A. Distinct reason from every staleness above.
approval_b = human_approval(action_b, "alice@ops (WebAuthn)", NOW, expires_at="2026-07-08T15:20:00Z")
substituted = agent_receipt(action, approval_b, NOW)   # executing `action`, ref -> approval of action_b
print("digest-substitution ->", verify_chain(action, approval_b, substituted, NOW))

tamper              -> (False, 'digest substitution: the approval binds a different canonical action than the one being executed')
confused-deputy     -> (False, 'digest substitution: the approval binds a different canonical action than the one being executed')
replay              -> (False, 'approval already consumed (replay refused)')
forged-approval     -> (False, 'approval: signature invalid (forged, tampered, or malformed)')
self-minted-agent   -> (False, 'agent receipt: signature invalid (forged, tampered, or malformed)')
wrong-action-agent  -> (False, 'digest substitution: the agent receipt binds a different canonical action than the one being executed')
malformed-approval  -> (False, 'approval: receipt malformed (not an object with a signature object)')
malformed-agent     -> (False, 'agent receipt: receipt malformed (not an object with a signature object)')
wrong-len-sig       -> (False, 'approval: signature invalid (forged, tampered, or malformed)')
nonobject-receipt   -> (Fa

## یہ کیا ثابت کرتا ہے — اور کیا نہیں کرتا

**ثابت کرتا ہے:** ایک نامزد انسان نے اس *بالکل مستند عمل* کی منظوری دی (مکمل عمل + ہضم، ایک کلید سے دستخط شدہ جو ایک پِن کی گئی رجسٹری سے حل کی گئی ہے)، اور ایجنٹ نے *بالکل وہی منظور شدہ عمل* انجام دیا (وہی ہضم، رسید منظوری سے `receipt_hash` کے ذریعے منسلک، سبق کی اپنی چین کنونشن) — جبکہ منظوری کی پالیسی کا ورژن، کلید، اور ختم ہونے کی تاریخ ابھی بھی موجود تھی، بالکل ایک بار۔ اگر دونوں میں سے کوئی بھی تبدیل ہو جائے، تو چین بند ہو جاتی ہے، اور انکار کی وجہ آپ کو بتاتی ہے **کہ کون سا** خاصیت ٹوٹ گئی: پرانی اتھارٹی بمقابلہ تبدیل شدہ عمل۔

**ثابت نہیں کرتا:** کہ منظوری کا یوزر انٹرفیس انسان کو وہی دکھا رہا تھا جو وہ دستخط کر رہا تھا (WYSIWYS ایک الگ مسئلہ ہے)، کہ چابی چور یا زبردستی حاصل نہیں کی گئی تھی پہلے سے ہی تبدیلی کے، یا کہ نیچے کے اثرات عمل سے میل کھاتے ہیں۔ دستخط شدہ ≠ منظور شدہ: ایک درست دستخط ایک پرانی پالیسی، تبدیل شدہ کلید، ختم شدہ مدت، یا مختلف ہضم پر یہاں کچھ فراہم نہیں کرتا۔

دونوں قسم کی رسیدیں سبق کے لفافے اور ایک `verify_chain` کوڈ راہ کو جان بوجھ کر شیئر کرتی ہیں: آپ نے جو بائنڈنگ ایکشن رسیدوں کے لیے مین نوٹ بک میں تیار کی ہے وہی کوڈ ہے جو انسان کی منظوری کو چیک کرتا ہے۔ ایک ویریفائر معاہدہ، الگ الگ پِن کی گئی اتھارٹیز، جو مستند عمل کے ہضم سے جُڑی ہیں اور کچھ نہیں۔


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ڈس کلیمر**:
یہ دستاویز AI ترجمہ سروس [Co-op Translator](https://github.com/Azure/co-op-translator) کے ذریعے ترجمہ کی گئی ہے۔ جبکہ ہم درستگی کے لیے کوشاں ہیں، براہ کرم اس بات سے آگاہ رہیں کہ خودکار ترجمے میں غلطیاں یا عدم درستیاں ہو سکتی ہیں۔ اصل دستاویز اپنے مادری زبان میں مستند ماخذ سمجھی جائے گی۔ حساس معلومات کے لیے پیشہ ور انسانی ترجمہ کی سفارش کی جاتی ہے۔ اس ترجمے کے استعمال سے پیدا ہونے والی کسی بھی غلط فہمی یا غلط تشریح کی ذمہ داری ہم قبول نہیں کرتے۔
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
